In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import branca.colormap as bcm
from shapely import wkt

from coordinate_to_ward import *      # ward_shape(), boundaries
from street_intersections import *    # the `intersections` table

In [ ]:
# Every Chicago crash with a recorded location, as points, clipped to the 27th Ward.
our_ward = ward_shape(27)

crashes = pd.read_csv('Traffic_Crashes_-_Crashes_20260622.csv', low_memory=False).dropna(subset=['LOCATION'])
crashes['LOCATION'] = crashes['LOCATION'].apply(wkt.loads)
crashes = gpd.GeoDataFrame(crashes, geometry='LOCATION', crs='EPSG:4326')
crashes = crashes[crashes.within(our_ward)]

print(f'{len(crashes):,} crashes in the 27th Ward')

In [ ]:
# Crashes the police flagged as intersection-related.
intersection_related = crashes[crashes['INTERSECTION_RELATED_I'] == 'Y'].copy()
print(f'{len(intersection_related):,} are intersection-related')

## Reusable map helpers

Small functions every map below shares, so each map cell stays a few lines with no copy-pasted folium boilerplate.

In [ ]:
# A folium map centred on the ward, with the ward outline already drawn.
WARD_CENTER = [our_ward.centroid.y, our_ward.centroid.x]

def ward_map(tiles='CartoDB positron'):
    m = folium.Map(location=WARD_CENTER, zoom_start=14, tiles=tiles)
    folium.GeoJson(
        our_ward.__geo_interface__, name='27th Ward boundary',
        style_function=lambda _: {'color': '#1f4eaa', 'weight': 2, 'fill': False},
    ).add_to(m)
    return m

# A 0..vmax YlOrRd colour scale, shown as the map legend.
def heat_scale(vmax, caption):
    scale = bcm.linear.YlOrRd_09.scale(0, vmax)
    scale.caption = caption
    return scale

# Draw one fixed-size circle per intersection, coloured by its 'value' column.
def intersection_map(data, caption, label):
    m = ward_map()
    scale = heat_scale(float(data['value'].max()), caption)
    scale.add_to(m)
    for _, r in data.iterrows():
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x], radius=7, weight=0,
            fill=True, fill_color=scale(r['value']), fill_opacity=0.9, popup=label(r),
        ).add_to(m)
    folium.LayerControl().add_to(m)
    return m

In [ ]:
# Intersection points (from street_intersections.py) that lie inside the ward.
ints = intersections.reset_index().dropna(subset=['lon', 'lat'])
ward_intersections = gpd.GeoDataFrame(
    ints, geometry=gpd.points_from_xy(ints.lon, ints.lat), crs='EPSG:4326',
)
ward_intersections = ward_intersections[ward_intersections.within(our_ward)].reset_index(drop=True)

# Aggregate crashes onto their nearest ward intersection.
# value=None -> crash count; otherwise the sum of that column. Empty intersections are dropped.
# Distances use EPSG:3435 (feet) so "nearest" is true ground distance, not degrees.
def by_intersection(points, value=None):
    nearest = gpd.sjoin_nearest(
        points.to_crs(3435), ward_intersections.to_crs(3435)[['geometry']]
    )['index_right']
    totals = nearest.value_counts() if value is None else points[value].groupby(nearest).sum()
    out = ward_intersections.assign(value=ward_intersections.index.map(totals).fillna(0))
    return out[out['value'] > 0]

## Crashes by intersection

Each intersection-related crash is mapped to the nearest ward intersection; circles are coloured by how many crashes land there.

In [ ]:
counts = by_intersection(intersection_related)
intersection_map(
    counts,
    'Intersection-related crashes (by nearest intersection)',
    lambda r: f"{r.streets}: {int(r['value'])} crashes",
)

## Population by 2020 Census block (official counts)

Map using exact **2020 Census** counts. The TIGER/Line `tl_2020_17_tabblock20` file embeds `POP20`/`HOUSING20` directly in each Illinois block polygon, so no separate population table is needed. Blocks are the finest official census geography, giving a sharper choropleth (people per km² of land) than the 100 m GHSL raster. The two totals (~57k vs ~53k) cross-check each other.

In [ ]:
# Census blocks near the ward (bbox filter keeps this fast vs. all 370k IL blocks),
# with 2020 population and people-per-km^2 density.
blocks = gpd.read_file('census/tl_2020_17_tabblock20.zip', bbox=tuple(our_ward.bounds)).to_crs('EPSG:4326')
blocks = blocks[blocks.intersects(our_ward)].copy()
blocks['POP20'] = blocks['POP20'].astype(int)
blocks['density'] = np.where(blocks['ALAND20'] > 0, blocks['POP20'] / (blocks['ALAND20'] / 1e6), 0).round().astype(int)

ward_pop = int(blocks.loc[blocks.representative_point().within(our_ward), 'POP20'].sum())
print(f'{len(blocks)} blocks | ward population (POP20): {ward_pop:,}')

# Clip block polygons to the ward outline and colour by density.
clipped = gpd.clip(blocks, our_ward)
scale = heat_scale(float(np.percentile(blocks.loc[blocks.density > 0, 'density'], 95)),
                   'People per km² (2020 Census blocks)')

m = ward_map()
folium.GeoJson(
    clipped[['geometry', 'GEOID20', 'POP20', 'density']],
    style_function=lambda f: {'fillColor': scale(min(f['properties']['density'], scale.vmax)),
                              'color': 'none', 'fillOpacity': 0.6},
    tooltip=folium.GeoJsonTooltip(['GEOID20', 'POP20', 'density'],
                                  aliases=['Block', 'Population', 'People / km²']),
    name='Population by block',
).add_to(m)
scale.add_to(m)
folium.LayerControl().add_to(m)
m

## Population density (GHSL 100 m raster)

A modelled population surface (GHS_POP, epoch 2025) for cross-checking the census blocks. Every tile in `GHS_POP/` is mosaicked, clipped to the ward, and shown as a smooth, semi-transparent heat layer.

In [ ]:
import glob, matplotlib
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
from rasterio.io import MemoryFile
from rasterio.warp import calculate_default_transform, reproject, Resampling
from scipy.ndimage import gaussian_filter

# Mosaic every tile in GHS_POP/ and clip to the ward (in the raster's Mollweide CRS).
srcs = [rasterio.open(t) for t in sorted(glob.glob('GHS_POP/GHS_POP_*.tif'))]
mosaic, mosaic_t = merge(srcs)
pop_crs, nodata, prof = srcs[0].crs, srcs[0].nodata, srcs[0].profile
for s in srcs:
    s.close()
prof.update(height=mosaic.shape[1], width=mosaic.shape[2], transform=mosaic_t)

ward_moll = gpd.GeoSeries([our_ward], crs='EPSG:4326').to_crs(pop_crs).iloc[0]
with MemoryFile() as mem:
    with mem.open(**prof) as ds:
        ds.write(mosaic)
    with mem.open() as ds:
        clip, clip_t = mask(ds, [ward_moll.__geo_interface__], crop=True, filled=True, nodata=nodata)

native = clip[0]
print(f"ward population (sum of 100 m cells): {native[(native != nodata) & (native > 0)].sum():,.0f}")

# Reproject the clip to lon/lat so it lines up with the basemap.
h, w = native.shape
left, top = clip_t.c, clip_t.f
dst_t, dw, dh = calculate_default_transform(
    pop_crs, 'EPSG:4326', w, h, left, top + h * clip_t.e, left + w * clip_t.a, top)
pop = np.full((dh, dw), nodata)
reproject(native, pop, src_transform=clip_t, src_crs=pop_crs, dst_transform=dst_t,
          dst_crs='EPSG:4326', resampling=Resampling.bilinear, src_nodata=nodata, dst_nodata=nodata)

# Smooth, gap-free fill: every in-ward cell coloured, semi-transparent so streets show through.
inside = pop != nodata
density = gaussian_filter(np.where(inside, np.clip(pop, 0, None), 0.0), sigma=0.8)
vmax = float(np.percentile(density[inside], 99))
rgba = matplotlib.colormaps['YlOrRd'](matplotlib.colors.Normalize(0, vmax, clip=True)(density))
rgba[..., 3] = np.where(inside, 0.6, 0.0)
bounds = [[dst_t.f + pop.shape[0] * dst_t.e, dst_t.c],
          [dst_t.f, dst_t.c + pop.shape[1] * dst_t.a]]

m = ward_map()
folium.raster_layers.ImageOverlay(rgba, bounds=bounds, opacity=1.0, name='Population density').add_to(m)
heat_scale(vmax, 'People per 100 m cell (GHS_POP E2025)').add_to(m)
folium.LayerControl().add_to(m)
m

## Weighted crash analysis data

Comprehensive cost per injury severity (KABCO), from the National Safety Council:
https://injuryfacts.nsc.org/all-injuries/costs/guide-to-calculating-costs/data-details/

| Severity | Cost |
| --- | --- |
| Death (K) | $2,050,000 |
| Disabling (A) | $174,000 |
| Evident (B) | $45,000 |
| Possible (C) | $28,000 |
| No injury observed (O) | $7,700 |
| Property damage only (per vehicle) | $6,600 |

In [ ]:
death = 2_050_000
disabling = 174_000
evident = 45_000
possible = 28_000
no_injury = 7_700
property_damage_only = 6_600

## Intersection-related crashes weighted by injury cost

The same nearest-intersection map as above, but each crash now contributes its **comprehensive injury cost** (the KABCO values above × the people hurt at each severity) instead of a count of 1. This surfaces the intersections with the most *severe* crashes, not just the most frequent.

*Property-damage-only cost is per vehicle, and this crash table has no vehicle count, so it is omitted — we weight by injury severity only.*

In [ ]:
# Comprehensive injury cost of each crash = people hurt at each severity x its KABCO cost.
COST = {
    'INJURIES_FATAL': death,                   # K
    'INJURIES_INCAPACITATING': disabling,      # A
    'INJURIES_NON_INCAPACITATING': evident,    # B
    'INJURIES_REPORTED_NOT_EVIDENT': possible, # C
    'INJURIES_NO_INDICATION': no_injury,       # O
}
intersection_related['cost'] = sum(intersection_related[col].fillna(0) * w for col, w in COST.items())

cost_by_intersection = by_intersection(intersection_related, value='cost')
intersection_map(
    cost_by_intersection,
    'Injury cost of intersection-related crashes ($)',
    lambda r: f"{r.streets}: ${r['value']:,.0f}",
)